In [1]:
import sys
import os

import os
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# Model Registry & Architecture
from registry.model_registry import MODEL_REGISTRY
from models import segFormer, STANet, swin_earlyfusion, BIT, Gswin_tac, UNet

# Dataset & Preprocessing
from datasets.dataLoader import create_dataloaders
from utils.preprocess import preprocess_to_npy

# 🔥 UPDATED LOSS FUNCTIONS 
from loss.ce_dice_loss import FocalDiceCELoss
from loss.boundary_loss import DeforestationTotalLoss


# Evaluation Metrics
from evaluation.metrics import compute_confusion_matrix, evaluate_all
from evaluation.evaluate import evaluate_model

# Utils & Configs
from configs import train_config as config
from utils.forward_model import forward_model
from logs.logger import Logger

/Users/macbook/Documents/Binus/Research Track/Codes/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Training modular code

In [ ]:
import torch
from torch.cuda.amp import autocast, GradScaler
import os

def train_model(
    model,
    train_loader,
    val_loader,
    config,
    criterion,
    optimizer,
    model_type,
    scheduler,
    device,
    logger
):

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    scaler = GradScaler()

    best_miou = 0
    os.makedirs(config.checkpoint_dir, exist_ok=True)

    for epoch in range(config.num_epochs):

        print(f"\n===== Epoch {epoch+1}/{config.num_epochs} =====")

        # ===============================
        # TRAIN
        # ===============================
        model.train()
        train_loss = 0
        
        # Dictionary untuk akumulasi detail loss per epoch
        epoch_loss_details = {"ce": 0, "focal": 0, "dice": 0, "boundary": 0}
        num_batches = 0

        for t1, t2, mask, file_id in train_loader:
            t1 = t1.to(device)
            t2 = t2.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            with autocast():
                pred = forward_model(model, t1, t2, model_type)
                # 🔥 FIX: Unpack tuple (total_loss, loss_details_dict)
                loss, loss_details = criterion(pred, mask)

            # 🔥 CHECK LOSS STABILITY
            if not torch.isfinite(loss):
                print(f"Invalid loss detected in: {file_id}")
                print(f"Loss details: {loss_details}")
                raise RuntimeError(f"Invalid loss at {file_id}")

            scaler.scale(loss).backward()

            # Gradient Clipping (Sangat disarankan untuk Hybrid Loss)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            # Akumulasi loss untuk monitoring
            train_loss += loss.item()
            for key in epoch_loss_details:
                if key in loss_details:
                    epoch_loss_details[key] += loss_details[key]
            
            num_batches += 1

        # Rata-rata Loss per Epoch
        train_loss /= num_batches
        for key in epoch_loss_details:
            epoch_loss_details[key] /= num_batches

        print(f"Train Loss: {train_loss:.4f} (Focal: {epoch_loss_details['focal']:.4f}, Dice: {epoch_loss_details['dice']:.4f})")

        # ===============================
        # VALIDATION
        # ===============================
        model.eval()
        val_loss = 0
        num_val_batches = 0

        conf_matrix = torch.zeros((config.num_classes, config.num_classes)).to(device)

        with torch.no_grad():
            for t1, t2, mask, file_id in val_loader:
                t1 = t1.to(device)
                t2 = t2.to(device)
                mask = mask.to(device)

                with autocast():
                    pred = forward_model(model, t1, t2, model_type)
                    # 🔥 FIX: Unpack tuple di validation juga
                    loss, _ = criterion(pred, mask)

                val_loss += loss.item()
                num_val_batches += 1

                pred_label = torch.argmax(pred, dim=1)
                conf_matrix += compute_confusion_matrix(pred_label, mask, config.num_classes)

        val_loss /= num_val_batches

        # ===============================
        # LOGGER & METRICS
        # ===============================
        results = evaluate_all(conf_matrix)
        
        # 1. Ekstrak miou utama
        miou = results.get("iou", 0) 
        
        # 2. Ekstrak array IoU per class (PASTIKAN baris ini ada)
        iou_per_class = results.get("_iou_per_class", [0, 0, 0]) 

        metrics = {
            "iou": miou,
            "f1": results.get("f1", 0),
            "precision": results.get("precision", 0),
            "recall": results.get("recall", 0),
            "accuracy": results.get("accuracy", 0)
        }

        if logger:
            logger.log(epoch=epoch + 1, train_loss=train_loss, val_loss=val_loss, metrics=metrics)
            logger.print(epoch=epoch + 1, train_loss=train_loss, val_loss=val_loss, metrics=metrics)
            
            # Sekarang iou_per_class sudah didefinisikan di atas, jadi bisa di-print
            print(f"   > IoU per Class:")
            print(f"     [Class 0 - No Change]: {iou_per_class[0]:.4f}")
            print(f"     [Class 1 - Deforest ]: {iou_per_class[1]:.4f}")
            print(f"     [Class 2 - Forest   ]: {iou_per_class[2]:.4f}")
            
            print(f"   > Detail Loss: CE: {epoch_loss_details['ce']:.4f} | Focal: {epoch_loss_details['focal']:.4f} | Dice: {epoch_loss_details['dice']:.4f}")

        # ===============================
        # SCHEDULER
        # ===============================
        if scheduler:
            try:
                scheduler.step(val_loss) # Untuk ReduceLROnPlateau
            except:
                scheduler.step()

        # ===============================
        # SAVE BEST MODEL
        # ===============================
        if miou > best_miou:
            best_miou = miou
            if config.save_model:
                path = os.path.join(config.checkpoint_dir, f"{config.model_name}_best.pth")
                torch.save(model.state_dict(), path)
                print(f"New Best mIoU: {best_miou:.4f}. Model saved!")

    print("Training finished.")

In [4]:
root_dir = "data/"
# cell 1 (sekali saja)
preprocess_to_npy(root_dir=root_dir)

2. Init DataLoader

In [5]:
train_loader, val_loader, test_loader = create_dataloaders(
    root_dir=root_dir,
    batch_size=config.TrainConfig.batch_size
)

print(len(train_loader))
print(len(val_loader))
print(len(test_loader))


72
9
9


3. Init parameters

In [6]:
# ====================================
# HITUNG CLASS WEIGHT (DARI TRAIN LABEL)
# ====================================
cache_dir = os.path.join(root_dir, "cache_npy")
labels = np.load(os.path.join(cache_dir, "labels.npy"))

indices = np.arange(len(labels))

train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    indices, labels, test_size=0.2, stratify=labels, random_state=42
)

# 🔥 Class weight otomatis (Raw Inverse Frequency)
ids = np.load(os.path.join(cache_dir, "ids.npy"))
num_classes = 3
class_counts = np.zeros(num_classes)

for idx in train_idx:
    file_id = ids[idx]
    npy_path = os.path.join(cache_dir, file_id.replace(".tif", ".npz"))
    data = np.load(npy_path)
    mask = data["mask"]

    for c in range(num_classes):
        # Menghitung pixel per kelas (Ignore index -1 tidak dihitung)
        class_counts[c] += (mask == c).sum()

# Normalisasi agar kelas terkecil tidak menyebabkan bobot terlalu raksasa sebelum smoothing
class_counts[class_counts == 0] = 1
total_pixels = class_counts.sum()
# Gunakan Raw Inverse Frequency (Tanpa sqrt di sini, karena sqrt dilakukan di dalam class Loss)
raw_weights = total_pixels / (num_classes * class_counts)
raw_weights = torch.tensor(raw_weights, dtype=torch.float32).to(config.TrainConfig.device)

# ====================================
# LOSS (MENGGUNAKAN UPDATE TERBARU)
# ====================================
# 1. Untuk model baseline (U-Net EF) menggunakan Focal + Dice + CE
criterion_model = FocalDiceCELoss(
    raw_weights=raw_weights,
    ce_weight=0.5,      # Sesuai diskusi: CE sebagai anchor unweighted (0.5)
    focal_weight=1.0, 
    dice_weight=1.0
)

# 2. Untuk model riset (GSwinT AC) menggunakan versi lengkap (dengan Boundary)
criterion_gswintac = DeforestationTotalLoss(  
    raw_weights=raw_weights,
    ce_weight=0.5,
    focal_weight=1.0,
    dice_weight=1.0,
    boundary_weight=0.5 # Sesuai diskusi: 0.5 agar tidak terlalu mendominasi
)


# ====================================
# MODEL & OPTIMIZER (TIDAK BERUBAH)
# ====================================
model_info = MODEL_REGISTRY["unet_ef"]
model = model_info["model"]()
model.to(config.TrainConfig.device) # Pastikan model masuk ke device
model_type = model_info["type"]
optimizer_name = model_info["optimizer"]

if optimizer_name == "Adam":
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.TrainConfig.learning_rate
    )
elif optimizer_name == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.TrainConfig.learning_rate,
        weight_decay=config.TrainConfig.weight_decay
    )

# ====================================
# SCHEDULER (SANGAT DISARANKAN UNTUK HYBRID LOSS)
# ====================================
scheduler = None
if config.TrainConfig.scheduler == "cosine":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config.TrainConfig.t_max
    )
# Tambahan: Scheduler ini sangat membantu menjaga stabilitas saat Dice & Focal berinteraksi

4. Train the models ( makesure to change model_name in config first )

In [20]:
if(config.TrainConfig.model_name != "gswin_tac") :
    criterion = criterion_model
else: criterion = criterion_gswintac

logger = Logger()

train_model(
    model= model,
    train_loader=train_loader,
    val_loader = val_loader,
    config=config.TrainConfig,
    criterion=criterion,
    optimizer= optimizer,
    model_type= model_type,
    scheduler= scheduler,
    device=config.TrainConfig.device,
    logger=logger
)

/var/folders/y4/8wf9ymkn149fmgb1hgtj41200000gn/T/ipykernel_3435/2919394260.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/Users/macbook/Documents/Binus/Research Track/Codes/venv/lib/python3.13/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/var/folders/y4/8wf9ymkn149fmgb1hgtj41200000gn/T/ipykernel_3435/2919394260.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/Users/macbook/Documents/Binus/Research Track/Codes/venv/lib/python3.13/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(



===== Epoch 1/30 =====
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0, 1, 2])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0, 1, 2])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tens

/var/folders/y4/8wf9ymkn149fmgb1hgtj41200000gn/T/ipykernel_3435/2919394260.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0, 1, 2])
tensor([0])
tensor([0, 1, 2])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])

Epoch 1
Train Loss : 1.3563
Val Loss   : 1.3982
IOU       : 0.4172
F1        : 0.4755
PRECISION : 0.0000
RECALL    : 0.0000
ACCURACY  : 0.0000
Detail Loss Component -> CE: 0.6848, Bnd: 0.0000
New Best mIoU: 0.4172. Mode

/var/folders/y4/8wf9ymkn149fmgb1hgtj41200000gn/T/ipykernel_3435/2919394260.py:146: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(val_loss) # Untuk ReduceLROnPlateau


tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0, 1, 2])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0])
tensor([0]

KeyboardInterrupt: 

5. TEST the models 

In [ ]:
model_path = os.path.join(
    config.TrainConfig.checkpoint_dir,
    f"{config.TrainConfig.model_name}_best.pth"
)

evaluate_model(
    model=model,
    loader=test_loader,
    model_path=model_path,
    num_classes=config.TrainConfig.num_classes,
    model_type=model_type,
    device=config.TrainConfig.device
)